# Notebook for assigning lava bedding orientations to NSVG pmag sites

In [13]:
import geopandas as gpd
import folium
import pandas as pd
import matplotlib.pyplot as plt
from shapely.geometry import Point, Polygon, LineString
from geopy.distance import geodesic
from pyhigh import get_elevation
import numpy as np
import pmagpy.ipmag as ipmag
from strat_height import *

%matplotlib inline
%config InlineBackend.figure_format = 'retina'

## Make handy functions

In [14]:
def estimate_height_from_strat_top(folium_map, site_data, lava_polygon, datum,
                                   distance=5000, lat_col='lat', lon_col='lon', 
                                   dip_col='bedding_dip', dip_dir_col='bedding_dip_dir', 
                                   color='purple', multipoint_selection='min'):
    '''
    function for estimating the strat height of the site from the top of a unit

    Parameters
    ----------
    folium_map : folium map object
        folium map object
    site_data : pandas DataFrame
        site data
    lava_polygon : shapely Polygon object
        lava polygon
    datum : float
        datum for the calculated strat height
        need to assign to be the basis of a lava flow as provided by Green, 2011
    distance : float
        distance in meters to draw the line from the site toward the lava base
        for calculating the intersection point
    lat_col : str
        column name for latitude in site_data
    lon_col : str
        column name for longitude in site_data
    dip_col : str
        column name for dip in site_data
    dip_dir_col : str
        column name for dip direction in site_data
    color : str
        color of the circle marker
        
    '''
    for i, r in site_data.iterrows():
        r[lon_col] = r[lon_col] % 360
        site_elevation = get_elevation(r[lat_col], (r[lon_col] + 180) % 360 - 180)
        line = create_line_from_point(r[lon_col], r[lat_col], r[dip_dir_col], distance=distance)
        intersection = line.intersection(lava_polygon.exterior)
        if intersection.type == 'Point':
            folium.CircleMarker([intersection.y, intersection.x], 
                                radius=5,  # Size of the circle marker
                                color="",  # Border color of the circle
                                fill=True,
                                fill_color=color,  # Fill color of the circle
                                fill_opacity=0.7).add_to(folium_map)
        elif intersection.type == 'MultiPoint':
            # there are multiple points, we want the one with smallest longitude in the MultiPoint object
            if multipoint_selection == 'min':
                intersection = min(intersection.geoms, key=lambda point: point.x)
            elif multipoint_selection == 'max':
                intersection = max(intersection.geoms, key=lambda point: point.x)
            folium.CircleMarker([intersection.y, intersection.x], 
                                radius=5,  # Size of the circle marker
                                color="",  # Border color of the circle
                                fill=True,
                                fill_color=color,  # Fill color of the circle
                                fill_opacity=0.7).add_to(folium_map)
        print('site lat', r[lat_col], 'site lon', r[lon_col], 'site elevation', site_elevation, 'intersection lat', intersection.y, 'intersection lon', intersection.x)
        
        intersection_elevation = get_elevation(intersection.y, (intersection.x + 180) % 360 - 180)
        folium.PolyLine(locations=[(r[lat_col], r[lon_col]), (intersection.y, intersection.x)], color=color).add_to(folium_map)
        
        # now calculate the distance in meters of the line from the intersection point to the site
        distance_from_flowbase = geodesic((r[lat_col], r[lon_col]), (intersection.y, intersection.x)).meters

        # site_data.loc[i, 'distance_from_flowbase'] = distance_from_flowbase
        # write the distance from the flow base to the site in the original site data table
        if intersection_elevation < site_elevation:
            site_data.loc[i, 'relative height'] = (distance_from_flowbase - (site_elevation - intersection_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
            site_data.loc[i, 'height'] = datum - (distance_from_flowbase - (site_elevation - intersection_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
        else:
            site_data.loc[i, 'relative height'] = (distance_from_flowbase + np.abs(intersection_elevation - site_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))
            site_data.loc[i, 'height'] = datum - (distance_from_flowbase + np.abs(intersection_elevation - site_elevation) / np.tan(np.radians(r[dip_col]))) * np.sin(np.radians(r[dip_col]))

## Assign strat height for Diehl and Haig, 1994 flows in the Copper Harbor Conglomerate

- manually assigned the Lt sites to be the lowest flows at total strat height of 5100 m 
- assigned the LST sites to be evenly spaced through the middle lavas that are about 600 meters thick in total (600/20=30 meter spacing)
- assigned the OT sites to be the highest flows at total strat height of 5900 m

## Assign strat height for Books, 1972 flows
- most strat heights are presented in a strat diagram in the original paper
- the current heights are estimated based on digitizing the diagram

## Assign strat height for Hnat et al., 2006 data

- Let's hang the sites from the base of the Copper City Conglomerate

## Assign strat height for Kulakov et al., 2013 data

- their LST sites are from the middle Lake Shore Trap flows
- there is no useful strat info in the publication so we just assign the same height to them all--5600 meters

## Load bedding data

- from Cannon and Nicholson, 2001

In [15]:
PLV_SW_bedding = pd.read_csv('../data/GIS/Keweenaw/PLV_SW_bedding.csv')
PLV_SW_bedding['dip_dir'] = PLV_SW_bedding['strike'] + 90
PLV_middle_bedding = pd.read_csv('../data/GIS/Keweenaw/PLV_middle_bedding.csv')
PLV_middle_bedding['dip_dir'] = PLV_middle_bedding['strike'] + 90
PLV_NE_bedding = pd.read_csv('../data/GIS/Keweenaw/PLV_NE_bedding.csv')
PLV_NE_bedding['dip_dir'] = PLV_NE_bedding['strike'] + 90

## Load pmag data

- Hnat et al., 2006
- Founcher 2018 thesis work

In [16]:
Hnat2006_data = pd.read_csv('../data/pmag_compiled/Hnat2006/sites.txt', sep='\t', header=1)
Hnat2006_data = Hnat2006_data[Hnat2006_data['site'] != 'H_PL33'].reset_index(drop=True)

Hnat_SW_sites = ['H_PL32', 'H_PL31', 'H_PL30', 'H_PL29', 'H_PL28', 'H_PL26']
Hnat_middle_sites = ['H_PL25', 'H_PL19', 'H_PL24', 'H_PL12', 'H_PL08', 'H_PL01', 'H_PL02', 'H_PL03', 'H_PL04', 'H_PL04', 'H_PL06', 'H_PL11']
Hnat_NE_sites = ['H_PL13', 'H_PL16', 'H_PL07', 'H_PL21', 'H_PL17', 'H_PL15', 'H_PL20', 'H_PL10', 'H_PL14', 'H_PL23', 'H_PL22']

Hnat_SW = Hnat2006_data[Hnat2006_data['site'].isin(Hnat_SW_sites)]
Hnat_middle = Hnat2006_data[Hnat2006_data['site'].isin(Hnat_middle_sites)]
Hnat_NE = Hnat2006_data[Hnat2006_data['site'].isin(Hnat_NE_sites)]

Hnat_SW = assign_lava_bedding_to_sites(Hnat_SW, PLV_SW_bedding, dip_dir_col='dip_dir', dip_col='dip')
Hnat_middle = assign_lava_bedding_to_sites(Hnat_middle, PLV_middle_bedding, dip_dir_col='dip_dir', dip_col='dip')
Hnat_NE = assign_lava_bedding_to_sites(Hnat_NE, PLV_NE_bedding, dip_dir_col='dip_dir', dip_col='dip')

{'dec': 312.66414382971305, 'inc': 49.805804698429604, 'n': 21, 'r': 19.90867215206828, 'k': 18.326298589286374, 'alpha95': 7.631633869543636, 'csd': 18.921154840918152}
{'dec': 302.39833845252116, 'inc': 42.12819708834338, 'n': 42, 'r': 34.20649969666629, 'k': 5.26079404686262, 'alpha95': 10.663937792603658, 'csd': 35.31501286856123}
{'dec': 351.6450099056112, 'inc': 42.643885785560364, 'n': 46, 'r': 43.67854221740241, 'k': 19.384371465781012, 'alpha95': 4.902637020315623, 'csd': 18.397514633850108}


In [17]:
Foucher2018_data = pd.read_csv('../data/pmag_compiled/Foucher2018/sites.txt', sep='\t', header=1)
Foucher2018_Greenstone_data = Foucher2018_data[Foucher2018_data['dir_tilt_correction'] == 100]
Foucher2018_Greenstone_data = Foucher2018_data[Foucher2018_data['geologic_types'] == 'Lava flow'].reset_index(drop=True)
Foucher2018_Greenstone_data['dip'] = -Foucher2018_Greenstone_data['dip']
Foucher2018_Greenstone_data['dip_dir'] = (Foucher2018_Greenstone_data['dip_trend'] + 180) % 360
Foucher2018_Greenstone_orientation = ipmag.fisher_mean(Foucher2018_Greenstone_data['dip_dir'].tolist(), 
                                                       Foucher2018_Greenstone_data['dip'].tolist())
Foucher2018_Greenstone_data['bedding_dip'] = Foucher2018_Greenstone_orientation['inc']
Foucher2018_Greenstone_data['bedding_dip_dir'] = Foucher2018_Greenstone_orientation['dec']

## load Copper Harbor Conglomerate polygons

In [18]:
CHC_polygons = gpd.read_file("../data/GIS/Keweenaw/Copper_Harbor_Conglomerate.shp")
CHC_polygons = CHC_polygons.to_crs(epsg=4326)
# make sure all the polygons longitudes are between 0 and 360
CHC_polygons = CHC_polygons.explode(ignore_index=True)
CHC_polygons['geometry'] = [Polygon([(x[0] % 360, x[1]) for x in poly.exterior.coords]) for poly in CHC_polygons['geometry']]
CHC_polygons = CHC_polygons.unary_union

## load Greenstone Flow polygon

In [19]:
Greenstone_Flow = gpd.read_file("../data/GIS/Keweenaw/Greenstone_Flow.shp")
Greenstone_Flow = Greenstone_Flow.to_crs(epsg=4326)
Greenstone_Flow = Greenstone_Flow.explode(ignore_index=True)
Greenstone_Flow['geometry'] = [Polygon([(x[0] % 360, x[1]) for x in poly.exterior.coords]) for poly in Greenstone_Flow['geometry']]
Greenstone_Flow

,UNITNAME_P,ROCKTYPE_P,STRAT_P,ERA_P,PROVINCE_P,TECTZONE_P,AREA,PERIMETER,TYPE,GCM_CODE,...,WIGEOL_DD_,WIGEOL_DD1,ORIG_LABEL,SGMC_LABEL,UNIT_LINK,SOURCE,UNIT_AGE,ROCKTYPE1,ROCKTYPE2,geometry
0,None,None,None,None,None,None,6.999021e+07,1.934293,None,None,...,None,None,Ygr,Yplv;0,MIYplv;0,MI004,Middle Proterozoic,basalt,andesite,"POLYGON ((271.43187 47.14183, 271.43462 47.144..."


## Make the map

- the southwest section volcanics is 0.78 of the Calumet section
- the northeast section volcanics is 1.25 of the Calumet section

In [20]:
# plot the sites
CHC_map = folium.Map(location=[Hnat2006_data['lat'].mean(), Hnat2006_data['lon'].mean()%360], 
                      zoom_start=9, tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
    attr="Esri")

plot_lava(CHC_map, CHC_polygons.geoms[0], 'red')
plot_lava(CHC_map, CHC_polygons.geoms[1], 'green')
plot_lava(CHC_map, CHC_polygons.geoms[2], 'blue')
plot_lava(CHC_map, CHC_polygons.geoms[3], 'orange')
plot_lava(CHC_map, Greenstone_Flow.geometry[0], 'purple')

plot_site(CHC_map, Hnat_SW, 'orange')
plot_site(CHC_map, Hnat_middle, 'blue')
plot_site(CHC_map, Hnat_NE, 'green')
plot_site(CHC_map, Foucher2018_Greenstone_data, 'purple')

estimate_height_from_strat_top(CHC_map, Hnat_SW, CHC_polygons.geoms[3], 0.78*4613, distance=10000, 
                               lat_col='lat', lon_col='lon', color='orange', multipoint_selection='max')
estimate_height_from_strat_top(CHC_map, Hnat_middle, CHC_polygons.geoms[3], 4613, distance=10000, 
                               lat_col='lat', lon_col='lon', color='blue', multipoint_selection='max')
estimate_height_from_strat_top(CHC_map, Hnat_NE, CHC_polygons.geoms[3], 1.25*4613, distance=10000,
                                 lat_col='lat', lon_col='lon', color='green', multipoint_selection='max')

estimate_height_from_strat_top(CHC_map, Foucher2018_Greenstone_data, Greenstone_Flow.geometry[0], 8000*0.304, distance=10000,
                                 lat_col='lat', lon_col='lon', color='purple', multipoint_selection='max')
CHC_map

site lat 46.93 site lon 271.17 site elevation 451.0 intersection lat 46.94174800896473 intersection lon 271.157252807708
site lat 46.81 site lon 271.0 site elevation 377.0 intersection lat 46.823637561480474 intersection lon 270.98520254631165
site lat 46.78 site lon 270.95 site elevation 312.0 intersection lat 46.80000892609815 intersection lon 270.9282892889088
site lat 46.77 site lon 270.9 site elevation 363.0 intersection lat 46.78701338203867 intersection lon 270.8815396078573
site lat 46.7 site lon 270.77 site elevation 353.0 intersection lat 46.729751814066304 intersection lon 270.7377177557424
site lat 46.66 site lon 270.61 site elevation 348.0 intersection lat 46.717056307210534 intersection lon 270.5480909802104
site lat 47.28 site lon 271.59000000000003 site elevation 326.0 intersection lat 47.29526439212989 intersection lon 271.56594562492296
site lat 47.3 site lon 271.6 site elevation 260.0 intersection lat 47.3111833343522 intersection lon 271.58237675520706
site lat 47.3

## Save data

In [21]:
def save_data(file_name, site_data):
    # Write the extra row to the file first
    with open(file_name, 'w') as f:
        f.write('tab\tsite\n')  # Writing the extra row

    # Append the DataFrame to the file with the original headers
    site_data.to_csv(file_name, mode='a', index=False, sep='\t')

In [25]:
# Hnat PL32 strat height is difficult to estimate, let's use the same strat height as PL31
Hnat_SW.loc[Hnat_SW['site'] == 'H_PL32', 'height'] = Hnat_SW.loc[Hnat_SW['site'] == 'H_PL31', 'height'].values[0]

In [26]:
Hnat2006_with_height = pd.concat([Hnat_SW, Hnat_middle, Hnat_NE])
Foucher2018_Greenstone_data_with_height = Foucher2018_Greenstone_data

In [27]:
Hnat2006_with_height

,citations,dir_alpha95,dir_dec,dir_inc,dir_k,dir_n_samples,dir_polarity,dir_tilt_correction,geologic_classes,geologic_types,...,lon,method_codes,result_type,site,vgp_lat,vgp_lon,bedding_dip_dir,bedding_dip,relative height,height
23,This study,10.0,279.6,15.6,37.2,7,n,100,Extrusive,Lava Flow,...,-88.83,LT-T-Z,a,H_PL26,12.344790,179.630784,312.664144,49.805805,1255.884573,2342.255427
24,This study,2.9,288.8,42.9,321.8,9,n,100,Extrusive,Lava Flow,...,-89.00,LT-T-Z,a,H_PL28,30.480297,186.013210,312.664144,49.805805,1416.264509,2181.875491
25,This study,20.7,301.5,26.9,20.7,4,n,100,Extrusive,Lava Flow,...,-89.05,LT-T-Z,a,H_PL29,31.735687,167.300811,312.664144,49.805805,2162.258905,1435.881095
26,This study,13.5,301.0,-4.5,21.0,7,n,100,Extrusive,Lava Flow,...,-89.10,LT-T-Z,a,H_PL30,18.895450,155.762301,312.664144,49.805805,1781.261857,1816.878143
27,This study,9.2,268.0,36.0,43.6,7,n,100,Extrusive,Lava Flow,...,-89.23,LT-T-Z,a,H_PL31,13.061358,196.130501,312.664144,49.805805,3178.179587,419.960413
28,This study,16.7,301.9,20.4,31.4,4,n,100,Extrusive,Lava Flow,...,-89.39,LT-T-Z,a,H_PL32,29.309624,163.785941,312.664144,49.805805,6042.583863,419.960413
0,This study,7.8,302.8,40.8,51.9,8,n,100,Extrusive,Lava Flow,...,-88.41,LT-T-Z,a,H_PL01,38.942603,174.457358,302.398338,42.128197,1651.991445,2961.008555
1,This study,11.8,291.5,32.3,23.2,8,n,100,Extrusive,Lava Flow,...,-88.40,LT-T-Z,a,H_PL02,27.289363,178.217628,302.398338,42.128197,1223.337631,3389.662369
2,This study,7.0,274.3,45.7,63.4,8,n,100,Extrusive,Lava Flow,...,-88.38,LT-T-Z,a,H_PL03,22.359619,197.956681,302.398338,42.128197,1570.196927,3042.803073
3,This study,10.4,270.3,26.7,34.7,7,n,100,Extrusive,Lava Flow,...,-88.38,LT-T-Z,a,H_PL04,10.527256,191.081996,302.398338,42.128197,1570.196927,3042.803073


In [28]:
# save the site data
Hnat2006_file_name = '../data/pmag_compiled/Hnat2006/sites_with_height.txt'
Foucher2018_file_name = '../data/pmag_compiled/Foucher2018/sites_with_height.txt'
save_data(Hnat2006_file_name, Hnat2006_with_height)
save_data(Foucher2018_file_name, Foucher2018_Greenstone_data_with_height)